# 🚢 End-to-End Machine Learning Pipeline (v2 - Production-Ready)
### วิชา: CIT0013 Machine Learning | Lab 1: Titanic Survival Prediction

สมุดงาน (Notebook) นี้ถูกออกแบบมาสำหรับให้นักศึกษารันและเรียนรู้บน **Google Colab** เพื่อทำความเข้าใจขั้นตอนการสร้าง Machine Learning Pipeline ตั้งแต่เริ่มต้นจนถึงระดับพร้อมใช้งานจริง (Production)

#### 🌟 สิ่งที่นักศึกษาจะได้เรียนรู้ในเวอร์ชันนี้:
1. **การสร้าง Preprocessing Pipeline** ที่แยกจัดการข้อมูลตัวเลข (Numeric) และข้อมูลหมวดหมู่ (Categorical)
2. **การป้องกัน Data Leakage** ด้วยการรันกระบวนการเตรียมข้อมูลภายใน Pipeline เท่านั้น
3. **การคัดเลือกโมเดลที่ดีที่สุดแบบไดนามิก** และจูน Hyperparameters ด้วย GridSearchCV
4. **การประเมินผลอย่างเป็นระบบ** และถอดรหัสความสำคัญของฟีเจอร์ (Feature Importance) รองรับทัังโมเดลที่เป็นเชิงเส้น (Linear) และแบบต้นไม้ (Tree-based)
5. **การเตรียมโมเดลสำหรับนำไปใช้งานจริง (Production-Ready)** เช่น การเช็กความถูกต้องข้อมูลขาเข้า (Input Validation) และการเซฟโมเดลเก็บไว้ใช้งานต่อ

In [ ]:
# ติดตั้งไลบรารีที่จำเป็นสำหรับ Google Colab (หากรันในเครื่องส่วนตัวที่มีอยู่แล้วสามารถข้ามเซลล์นี้ได้)
!pip install seaborn pandas scikit-learn numpy joblib

---
## 🛠️ Step 1: นำเข้า Libraries และตั้งค่าระบบบันทึกการทำงาน (Logging)
ในงานระดับโปรดักชัน เรานิยมใช้ `logging` แทนการใช้ `print()` ทั่วไป เพื่อให้สามารถบันทึกเวลา ประเภทความรุนแรงของข้อความ และเขียนบันทึกเก็บเป็นไฟล์ (Log Files) ได้อย่างเป็นระบบ

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import logging
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

# ตั้งค่ารูปแบบแสดงข้อความ Log ใน Colab
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger("titanic-ml")

# ปิดข้อความ Log ที่ไม่จำเป็นจาก matplotlib และ seaborn
for noisy in ("matplotlib", "seaborn"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

log.info("Step 1: โหลด Libraries และตั้งค่า Logging เสร็จสิ้น")

---
## 📥 Step 2: โหลดข้อมูลจาก Seaborn
เราจะใช้ชุดข้อมูลผู้โดยสารเรือไททานิก (Titanic Dataset) ซึ่งเป็นข้อมูลจริงที่มีทั้งค่าสูญหาย (Missing values) และฟีเจอร์หลากหลายรูปแบบ

In [ ]:
# โหลดชุดข้อมูลของ seaborn เข้าสู่โครงสร้าง DataFrame ของ pandas
titanic = sns.load_dataset("titanic")

log.info(f"Step 2: โหลดข้อมูลสำเร็จ ขนาดข้อมูล={titanic.shape}")
log.info(f"รายชื่อฟีเจอร์ทั้งหมด: {list(titanic.columns)}")
print("\n--- ตัวอย่างข้อมูล 5 แถวแรก ---")
display(titanic[["survived", "pclass", "sex", "age", "fare", "embarked"]].head())

---
## 🔍 Step 3: ตรวจสอบข้อมูลเบื้องต้น (Quick Look)
ก่อนเริ่มลงมือทำโมเดล เราต้องเข้าใจคุณลักษณะของข้อมูลดิบและสแกนหาจุดข้อมูลที่ขาดหายไป

In [ ]:
log.info(f"อัตราการรอดชีวิตเฉลี่ยของผู้โดยสารทั้งหมด: {titanic['survived'].mean():.1%}")

print("\n--- จำนวนข้อมูลสูญหาย (Missing Values) ในแต่ละคอลัมน์ ---")
missing_summary = titanic.isnull().sum()
print(missing_summary[missing_summary > 0])

---
## ✂️ Step 4: เลือกคุณลักษณะ (Features) และแบ่งกลุ่ม Train/Test Set
เราจะแยกข้อมูลออกเป็นสองส่วน:
1. **Train Set (80%)** สำหรับให้โมเดลฝึกฝนการเรียนรู้
2. **Test Set (20%)** เก็บไว้สำหรับประเมินโมเดลในตอนสุดท้ายเท่านั้น ห้ามนำมาให้โมเดลเห็นในขณะฝึกสอนเด็ดขาด

*หมายเหตุ: เรากำหนด `stratify=y` เพื่อรักษาสัดส่วนของอัตราการรอดชีวิตให้ใกล้เคียงกันทั้งสองกลุ่ม*

In [ ]:
# ฟีเจอร์ที่ใช้ทำนาย: pclass (ระดับชั้นตั๋ว), sex (เพศ), age (อายุ), fare (ค่าตั๋ว), embarked (ท่าเรือที่ขึ้น)
feature_cols = ["pclass", "sex", "age", "fare", "embarked"]
X = titanic[feature_cols].copy()
y = titanic["survived"]

# ทำการแบ่งข้อมูลแบบ Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

log.info(f"จำนวนชุดฝึกสอน: {X_train.shape[0]} แถว | จำนวนชุดทดสอบ: {X_test.shape[0]} แถว")
log.info(f"อัตราการรอดในชุดฝึกสอน: {y_train.mean():.1%} | ในชุดทดสอบ: {y_test.mean():.1%}")

---
## 🤖 Step 5: สร้างกระบวนการเตรียมข้อมูลอัตโนมัติ (Preprocessing Pipeline)
ในขั้นตอนนี้เราจะแบ่งข้อมูลออกเป็น 2 ประเภทเพื่อจัดการด้วยท่อส่งข้อมูลย่อยที่แตกต่างกัน:
1. **ข้อมูลเชิงตัวเลข (Numeric Features):** `pclass`, `age`, `fare`
   * *Imputation:* เติมค่าว่างด้วยค่ามัธยฐาน (Median)
   * *Scaling:* ปรับช่วงของข้อมูลให้มีค่าเฉลี่ยเป็น 0 และความเบี่ยงเบนมาตรฐานเป็น 1 ด้วย `StandardScaler` เพื่อช่วยโมเดลเชิงเส้นเรียนรู้ได้ง่ายขึ้น
   * *เหตุผลด้านระเบียบวิธี:* เรานับ `pclass` (1, 2, 3) เป็น Numeric (เชิงอันดับ - Ordinal) เพราะชั้นตั๋วบอกทิศทางชัดเจน (1 ดีกว่า 2 และ 3) หากทำ One-Hot จะสูญเสียมิติด้านอันดับนี้ไป
2. **ข้อมูลเชิงหมวดหมู่ (Categorical Features):** `sex`, `embarked`
   * *Imputation:* เติมค่าว่างด้วยฐานนิยม (Mode หรือค่าที่พบบ่อยที่สุด) ด้วย `SimpleImputer(strategy='most_frequent')`
   * *One-Hot Encoding:* แปลงตัวอักษรเป็นเวกเตอร์เลขฐานสอง (0 และ 1)

In [ ]:
numeric_features = ["pclass", "age", "fare"]
categorical_features = ["sex", "embarked"]

# 1) ท่อสำหรับกลุ่มตัวเลข
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 2) ท่อสำหรับกลุ่มข้อมูลตัวอักษร / หมวดหมู่
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 3) รวมเข้าด้วยกันเป็น Preprocessor หลักด้วย ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

log.info("Step 5: สร้าง Preprocessor Pipeline เรียบร้อยพร้อมใช้งาน")

---
## 📊 Step 6: ทดสอบเปรียบเทียบโมเดลด้วย Cross-Validation
เราต้องการค้นหาว่าโมเดลสถาปัตยกรรมแบบใด (Logistic Regression, Decision Tree, หรือ Random Forest) ให้ผลลัพธ์เฉลี่ยดีที่สุด โดยการใช้ **5-fold Cross-Validation** เพื่อสับเปลี่ยนกลุ่มทดสอบการเรียนรู้ให้ทั่วถึง

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    # ประกอบโมเดลกับ Preprocessor เข้าด้วยกันเป็น Pipeline
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    # ประเมินคะแนนเฉลี่ยด้วย Cross Validation (5 ส่วนเท่าๆ กัน)
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    results[name] = scores
    log.info(f"  {name:20s}: คะแนนเฉลี่ย = {scores.mean():.3f} (+/- ส่วนเบี่ยงเบนเบื้องต้น {scores.std():.3f})")

# หาโมเดลที่ทำคะแนนได้ดีที่สุดโดยอัตโนมัติ
best_model_name = max(results, key=lambda k: results[k].mean())
best_mean = results[best_model_name].mean()
log.info(f"⭐ โมเดลที่ดีที่สุดจากการทดสอบคือ: {best_model_name} (คะแนนเฉลี่ย CV = {best_mean:.3f})")

---
## ⚙️ Step 7: การปรับค่าพารามิเตอร์แบบอัตโนมัติตามโมเดลที่ดีที่สุด (GridSearchCV)
เพื่อป้องกันข้อผิดพลาดและทำให้โค้ดมีความฉลาด โค้ดจะดึงโครงสร้างโมเดลที่ชนะใน Step 6 มาค้นหาพารามิเตอร์ที่ดีที่สุด (Hyperparameters Tuning) โดยกำหนดตัวเลือกช่วงจูนของแต่ละโมเดลเอาไว้ล่วงหน้า

In [ ]:
# ตารางทางเลือกในการจูนของแต่ละโมเดลหลัก
param_grids = {
    "Logistic Regression": {
        "model__C":       [0.01, 0.1, 1.0, 10.0],
        "model__penalty": ["l2"],
        "model__solver":  ["lbfgs"],
    },
    "Decision Tree": {
        "model__max_depth":         [None, 5, 10, 15],
        "model__min_samples_split": [2, 5, 10],
        "model__criterion":         ["gini", "entropy"],
    },
    "Random Forest": {
        "model__n_estimators":      [50, 100, 200],
        "model__max_depth":         [None, 5, 10],
        "model__min_samples_split": [2, 5],
    },
}

best_model_class = type(models[best_model_name])
selected_grid = param_grids.get(best_model_name)

# กรณีที่เป็นโมเดลอื่นๆ นอกเหนือการควบคุมจะเลือกใช้ Default Grid ของ Random Forest
if selected_grid is None:
    log.warning(f"ไม่พบชุดจูนของโมเดล {best_model_name} — จะใช้ Default ของ Random Forest")
    selected_grid = param_grids["Random Forest"]
    best_model_class = RandomForestClassifier

log.info(f"โมเดลที่กำลังค้นหาค่าพารามิเตอร์: {best_model_name}")
log.info(f"ตารางทางเลือกพารามิเตอร์ (Param Grid): {selected_grid}")

# สร้าง Pipeline สุดท้ายที่ดึงประเภทโมเดลที่ดีที่สุดมาทำงานแบบไดนามิก
final_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_model_class(random_state=42))
])

grid_search = GridSearchCV(
    final_pipe, selected_grid, cv=5,
    scoring="accuracy", n_jobs=-1
)
grid_search.fit(X_train, y_train)

log.info(f"ค่าพารามิเตอร์ที่ดีที่สุด (Best params): {grid_search.best_params_}")
log.info(f"คะแนนความถูกต้องจากการจูน (Best CV accuracy): {grid_search.best_score_:.3f}")

---
## 🎯 Step 8: ประเมินผลบนชุดทดสอบ (Test Set)
เมื่อได้โมเดลรุ่นที่ดีที่สุดแล้ว เราจะให้โมเดลทำนายข้อมูลชุดทดสอบ (Test Set) ที่แยกไว้ตั้งแต่ต้น เพื่อวัดผลและคำนวณค่า Precision, Recall และ F1-Score ในการทดสอบนี้

In [ ]:
# ดึงโมเดลและท่อส่งข้อมูลที่จูนเสร็จสมบูรณ์ออกมาใช้งาน
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
log.info(f"คะแนนความถูกต้องบน Test Set: {test_accuracy:.3f}")

# รายงานผลสถิติแบบละเอียด
cm = confusion_matrix(y_test, y_pred)
print("\n--- Confusion Matrix ---")
print(f"  เสียชีวิตจริง (True Negative):  TN = {cm[0,0]:3d} | ทำนายรอดแต่จริงเสีย (False Positive):  FP = {cm[0,1]:3d}")
print(f"  รอดจริงแต่ทำนายเสีย (False Negative): FN = {cm[1,0]:3d} | รอดชีวิตจริง (True Positive):   TP = {cm[1,1]:3d}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["เสียชีวิต", "รอดชีวิต"]))


---
## 💡 Step 9: ดึงระดับความสำคัญของปัจจัยที่ส่งผล (Feature Importance)
เพื่อให้เข้าใจได้ว่าฟีเจอร์ใดเป็นตัวแปรสำคัญต่อการตัดสินใจรอดชีวิตมากที่สุด เราจะดึงความสำคัญมานำเสนอ
* *การแก้ไขให้มีคุณภาพสูง:* หากเป็นโมเดลประเภท Tree-based จะสแกนผ่านคุณสมบัติ `.feature_importances_` หากเป็นกลุ่ม Linear (เช่น Logistic Regression) จะดึงน้ำหนักสัมประสิทธิ์สัมบูรณ์ผ่านคุณสมบัติ `.coef_` และปรับใช้ฟังก์ชันเฉลี่ยให้รอบรับกรณีโมเดลที่มีหลายคลาสในอนาคตได้อย่างปลอดภัย

In [ ]:
model_step = best_model["model"]
onehot_step = best_model["preprocessor"].named_transformers_["cat"]["onehot"]

# สร้างรายชื่อฟีเจอร์ใหม่ที่ขยายขนาดหลังจากการทำ One-Hot Encoding สำเร็จ
feature_names = numeric_features + list(onehot_step.get_feature_names_out(categorical_features))

# ตรวจหาคุณลักษณะด้านความสำคัญของแต่ละสถาปัตยกรรมโมเดล
if hasattr(model_step, "feature_importances_"):
    importances = model_step.feature_importances_
    log.info("ถอดรหัสฟีเจอร์ด้วยคุณสมบัติ: .feature_importances_ (Tree-based Model)")
elif hasattr(model_step, "coef_"):
    # คำนวณค่าเฉลี่ยของน้ำหนักสัมบูรณ์ (axis=0) เพื่อให้รองรับกรณีโมเดลแบบ Multiclass ในอนาคต
    importances = np.mean(np.abs(model_step.coef_), axis=0)
    log.info("ถอดรหัสฟีเจอร์ด้วยคุณสมบัติ: |.coef_| (Linear Model)")
else:
    log.warning("โมเดลสถาปัตยกรรมนี้ไม่รองรับการดึงค่าระดับความสำคัญเชิงคณิตศาสตร์")
    importances = None

if importances is not None:
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)

    print("\n--- ตารางจัดอันดับความสำคัญของฟีเจอร์ ---")
    print(importance_df.to_string(index=False))

---
## 📦 Step 9.5: เครื่องมือสำหรับใช้งานจริง (Production Tools - Validation & Save)
ในโปรดักชัน เราจะไม่ทำกระบวนการเทรนใหม่ซ้ำในทุกลำดับการทำนาย เรานิยมเซฟตัวแปรของ Pipeline ทั้งหมดไว้ และโหลดกลับมาใช้ทำนายข้อมูลหน้างานโดยตรง ตลอดจนเช็กข้อผิดพลาดของข้อมูลขาเข้าก่อนใช้งานจริง

In [ ]:
MODEL_FILE = Path("titanic_best_model.joblib")

# 1) ฟังก์ชันบันทึกตัวแปรโมเดล
def save_ml_pipeline(model, file_path: Path = MODEL_FILE) -> None:
    joblib.dump(model, file_path)
    log.info(f"บันทึกโมเดลไว้เรียบร้อยที่: {file_path.resolve()}")

# 2) ฟังก์ชันโหลดตัวแปรโมเดลกลับมาใช้งาน
def load_ml_pipeline(file_path: Path = MODEL_FILE):
    if not file_path.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์โมเดลที่ {file_path} กรุณาทำการเทรนก่อนรัน")
    return joblib.load(file_path)

# 3) ระบบตรวจเช็กข้อผิดพลาดและความสมบูรณ์ของโครงสร้างข้อมูลขาเข้า
REQUIRED_COLS = ["pclass", "sex", "age", "fare", "embarked"]
EXPECTED_PCLASS = {1, 2, 3}
EXPECTED_SEX = {"male", "female"}
EXPECTED_EMBARK = {"S", "C", "Q"}

def validate_passenger_input(df: pd.DataFrame) -> None:
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"ข้อมูลต้องนำเสนอในรูปแบบ DataFrame ของ pandas เท่านั้น | คลาสปัจจุบัน: {type(df).__name__}")

    missing = set(REQUIRED_COLS) - set(df.columns)
    if missing:
        raise ValueError(f"โครงสร้างตารางข้อมูลขาดคอลัมน์สำคัญ: {sorted(missing)}")

    for col in ["pclass", "age", "fare"]:
        if not pd.api.types.is_numeric_dtype(df[col]):
            raise TypeError(f"คอลัมน์ '{col}' ต้องเป็นประเภทตัวเลขเท่านั้น")

    bad_pclass = set(df["pclass"].dropna().unique()) - EXPECTED_PCLASS
    if bad_pclass:
        raise ValueError(f"pclass ต้องมีค่าเฉพาะ 1, 2 หรือ 3 เท่านั้น | ตรวจพบค่าแปลกปลอม: {bad_pclass}")

    bad_sex = set(df["sex"].dropna().unique()) - EXPECTED_SEX
    if bad_sex:
        raise ValueError(f"sex ต้องมีค่าเฉพาะ male หรือ female เท่านั้น | ตรวจพบค่าแปลกปลอม: {bad_sex}")

    bad_embarked = set(df["embarked"].dropna().unique()) - EXPECTED_EMBARK
    if bad_embarked:
        raise ValueError(f"embarked ต้องมีค่าระบุเฉพาะ S, C, หรือ Q เท่านั้น | ตรวจพบค่าแปลกปลอม: {bad_embarked}")

    # แจ้งเตือนเรื่องการตรวจเจอค่าว่าง และอธิบายว่าระบบเติมค่าว่างให้อัตโนมัติด้วย Pipeline
    if df[REQUIRED_COLS].isnull().any().any():
        null_info = df[REQUIRED_COLS].isnull().sum()
        log.info(f"⚠️ ตรวจพบข้อมูลสูญหาย (Missing values) ในข้อมูลใหม่ — Imputer จะเติมคำให้อัตโนมัติ:\n{null_info[null_info > 0].to_string()}")

# เซฟโมเดลที่ฝึกสำเร็จ
save_ml_pipeline(best_model)
log.info("โมเดลและเครื่องมือเตรียมพร้อมสำหรับการรันทำนายในขั้นตอนถัดไป")

---
## 🔮 Step 10: ทำนายผลลัพธ์ข้อมูลผู้โดยสารกลุ่มใหม่
เราจำลองข้อมูลจำลองของผู้โดยสาร 3 คน เพื่อให้ส่งผ่านโมเดลประมวลผลทำนายโอกาสความน่าจะเป็นในการรอดชีวิต

In [ ]:
# ข้อมูลจำลองผู้โดยสารใหม่ที่จะส่งเข้าทำนาย
new_passengers = pd.DataFrame({
    "pclass":   [1, 3, 2],
    "sex":      ["female", "male", "female"],
    "age":      [25, 30, 40],
    "fare":     [80.0, 7.5, 20.0],
    "embarked": ["C", "S", "Q"]
})

# 1) ทำการรันตรวจทานความถูกต้องโครงสร้าง (Validation)
try:
    validate_passenger_input(new_passengers)
except (TypeError, ValueError) as error:
    log.error(f"ข้อมูลนำเข้าไม่ผ่านเกณฑ์ตรวจสอบ: {error}")
    raise

# 2) ส่งโมเดลรันคำนวณและแสดงความน่าจะเป็น
predictions = best_model.predict(new_passengers)
probabilities = best_model.predict_proba(new_passengers)[:, 1]

print("\n--- ข้อมูลนำเข้าของกลุ่มใหม่ ---")
display(new_passengers)

print("\n--- ผลการรันทำนายจากโมเดล ---")
for idx, (pred, prob) in enumerate(zip(predictions, probabilities), start=1):
    result_text = "รอดชีวิต (Survived)" if pred == 1 else "เสียชีวิต (Deceased)"
    log.info(f"ผู้โดยสารอันดับ {idx}: {result_text} | โอกาสรอดโดยประมาณ = {prob:.1%}")

---
## 🔄 ส่วนเสริม (Bonus): โหลดโมเดลกลับมาใช้งานโดยตรง (ไม่ต้องรันฝึกใหม่ซ้ำ)
ตัวอย่างสำหรับการจำลองนำไปขึ้นเซิร์ฟเวอร์หรือนำไปเขียนร่วมกับ API: เราเพียงดึงไฟล์ `.joblib` กลับเข้ามาสู่ระบบแล้วรันผลทำนายได้ทันที

In [ ]:
log.info("กำลังเปิดดึงไฟล์โมเดลจากพื้นที่จัดเก็บ...")
loaded_pipeline = load_ml_pipeline()

# ใช้โมเดลที่โหลดมาทดสอบทำนาย
loaded_predictions = loaded_pipeline.predict(new_passengers)

log.info(f"ผลลัพธ์ของโมเดลที่โหลด: {loaded_predictions.tolist()}")
log.info(f"การตรวจสอบผลลัพธ์ความถูกต้องเทียบเคียงกับโมเดลเดิม: {loaded_predictions.tolist() == predictions.tolist()}")

log.info("🎉 เสร็จสิ้นกระบวนการ Machine Learning Pipeline v2 เรียบร้อยครบถ้วนทุกขั้นตอน!")